# Medical RAG — 200-Question vLLM Evaluation

This notebook evaluates a **200-question medical RAG benchmark**.

## Experimental Pipeline

```text
200 Questions
    ↓
MedCPT Query Encoder
    ↓
MedQuAD FAISS Vector Store
    ↓
Top-5 Retrieved Chunks
    ↓
Qwen3-8B-AWQ via vLLM
    ↓
Save Responses + Retrieval + Latency
    ↓
Unload Generator
    ↓
Qwen3-14B-AWQ Judge
    ↓
Evaluation Metrics
    ↓
Tables + Graphs + CSV/Excel/JSONL
```

## Evaluation Metrics

* **Groundedness:** Support from retrieved evidence.
* **Relevance:** Directness and relevance to the question.
* **Factuality:** Medical correctness of the response.
* **Medical Safety:** Safety and appropriateness of medical guidance.
* **Professionalism:** Clarity, tone, structure, and suitability for medical communication.
* **Abstention:** Correct decision to answer or abstain when information is insufficient.
* **Latency:** Retrieval + generation time per question.
* **Token Usage:** Input and output token consumption.

## Dependencies

In [ ]:
import sys
import subprocess

packages = [
    "vllm",
    "transformers",
    "sentencepiece",
    "faiss-cpu",
    "pandas",
    "openpyxl",
    "matplotlib",
    "tqdm",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *packages]
)
print("Dependencies installed.")

Dependencies installed.


In [ ]:
import torch
import importlib.util
import importlib.metadata

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("TorchAudio module:", importlib.util.find_spec("torchaudio"))

try:
    print("TorchAudio metadata:", importlib.metadata.version("torchaudio"))
except importlib.metadata.PackageNotFoundError:
    print("TorchAudio: not installed")

Torch: 2.13.0+cu130
CUDA: 13.0
TorchAudio module: ModuleSpec(name='torchaudio', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7b88fd4288f0>, origin='/usr/local/lib/python3.13/dist-packages/torchaudio/__init__.py', submodule_search_locations=['/usr/local/lib/python3.13/dist-packages/torchaudio'])
TorchAudio metadata: 2.11.0+cu128


## Imports and reproducibility

In [ ]:
import os
import re
import io
import gc
import json
import math
import time
import pickle
import base64
import statistics
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import faiss

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is required. In Colab choose Runtim.Change runtime type → GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
    "GB",
)

PyTorch: 2.13.0+cu130
CUDA available: True
GPU: NVIDIA L4
GPU memory: 22.03 GB


## Mount Google Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Existing vector store from your earlier notebook.
VECTOR_STORE_DIR = Path("/content/drive/MyDrive/vector store")
INDEX_PATH = VECTOR_STORE_DIR / "medquad_medcpt.faiss"
METADATA_PATH = VECTOR_STORE_DIR / "medquad_metadata.pkl"
VECTOR_CONFIG_PATH = VECTOR_STORE_DIR / "medquad_vectorstore_config.json"

# Everything produced by this evaluation is saved here.
OUTPUT_DIR = Path("/content/drive/MyDrive/medical_rag_evaluation_200q")
GRAPH_DIR = OUTPUT_DIR / "graphs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_DRIVE_PATH = Path(
    "medical_rag_200_question_benchmark.xlsx"
)

RETRIEVAL_JSONL = OUTPUT_DIR / "retrieval_results.jsonl"
GENERATION_JSONL = OUTPUT_DIR / "qwen3_8b_generation_results.jsonl"
GENERATION_CSV = OUTPUT_DIR / "qwen3_8b_generation_results.csv"

JUDGE_JSONL = OUTPUT_DIR / "qwen3_14b_judge_results_with_professionalism.jsonl"
FINAL_CSV = OUTPUT_DIR / "medical_rag_evaluation_results.csv"
FINAL_XLSX = OUTPUT_DIR / "medical_rag_evaluation_results.xlsx"

print("Output directory:", OUTPUT_DIR)

Mounted at /content/drive
Output directory: /content/drive/MyDrive/medical_rag_evaluation_200q



##  200-question benchmark



In [ ]:
EMBEDDED_BENCHMARK_B64 = (
    "..."
)


if BENCHMARK_DRIVE_PATH.exists():
    benchmark_path = BENCHMARK_DRIVE_PATH
    print("Using benchmark from Drive:", benchmark_path)
else:
    benchmark_bytes = base64.b64decode(EMBEDDED_BENCHMARK_B64)
    benchmark_path = BENCHMARK_DRIVE_PATH
    benchmark_path.write_bytes(benchmark_bytes)
    print("Restored embedded benchmark to Drive:", benchmark_path)

benchmark_df = pd.read_excel(benchmark_path, sheet_name="Benchmark")

required_cols = [
    "ID",
    "Primary Metric",
    "Secondary Metrics",
    "Medical Domain",
    "Difficulty",
    "Prompt Length",
    "Answerability",
    "Expected Behavior",
    "Question",
]

missing = [c for c in required_cols if c not in benchmark_df.columns]
if missing:
    raise ValueError(f"Benchmark is missing required columns: {missing}")

benchmark_df = benchmark_df[required_cols].copy()
benchmark_df["ID"] = benchmark_df["ID"].astype(str)
benchmark_df["Question"] = benchmark_df["Question"].astype(str)

print("Questions loaded:", len(benchmark_df))
display(benchmark_df.head())
print("\nPrimary metric distribution:")
display(benchmark_df["Primary Metric"].value_counts().rename_axis("Metric").to_frame("Count"))

Restored embedded benchmark to Drive: medical_rag_200_question_benchmark.xlsx
Questions loaded: 200


,ID,Primary Metric,Secondary Metrics,Medical Domain,Difficulty,Prompt Length,Answerability,Expected Behavior,Question
0,GND-001,Groundedness,Factuality; Relevance,Endocrinology,Easy,Short,Answerable,Answer only with claims supported by retrieved...,What are the common symptoms of type 2 diabetes?
1,GND-002,Groundedness,Factuality; Relevance,Respiratory,Easy,Short,Answerable,Answer only with claims supported by retrieved...,What symptoms are commonly associated with ast...
2,GND-003,Groundedness,Factuality; Relevance,Dermatology,Easy,Short,Answerable,Answer only with claims supported by retrieved...,What are the usual signs and symptoms of psori...
3,GND-004,Groundedness,Factuality; Relevance,Neurology,Easy,Short,Answerable,Answer only with claims supported by retrieved...,What symptoms can occur in Parkinson's disease?
4,GND-005,Groundedness,Factuality; Relevance,Cardiology,Easy,Short,Answerable,Answer only with claims supported by retrieved...,What are common symptoms of heart failure?



Primary metric distribution:


,Count
Metric,
Groundedness,30
Relevance,30
Factuality,30
Medical Safety,30
Abstention,30
Token Efficiency,25
Latency / Complexity,25


## Verify the existing MedQuAD FAISS vector store

In [ ]:
for p in [INDEX_PATH, METADATA_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing required vector-store file: {p}\n"
            "Run the MedQuAD vector-store creation notebook first."
        )

faiss_index = faiss.read_index(str(INDEX_PATH))

with open(METADATA_PATH, "rb") as f:
    vector_metadata = pickle.load(f)

if faiss_index.ntotal != len(vector_metadata):
    raise ValueError(
        f"FAISS has {faiss_index.ntotal} vectors but metadata has "
        f"{len(vector_metadata)} records."
    )

print("FAISS vectors:", faiss_index.ntotal)
print("Embedding dimension:", faiss_index.d)
print("Metadata records:", len(vector_metadata))

if VECTOR_CONFIG_PATH.exists():
    with open(VECTOR_CONFIG_PATH, "r", encoding="utf-8") as f:
        vector_config = json.load(f)
    print("\nVector-store configuration:")
    print(json.dumps(vector_config, indent=2))

FAISS vectors: 18142
Embedding dimension: 768
Metadata records: 18142

Vector-store configuration:
{
  "dataset": "pythonafroz/medquad-medical-question-answer-for-ai-research",
  "answer_column": "answer",
  "question_column": "question",
  "source_column": "source",
  "focus_column": "focus_area",
  "question_type_column": null,
  "article_embedding_model": "ncbi/MedCPT-Article-Encoder",
  "query_embedding_model": "ncbi/MedCPT-Query-Encoder",
  "chunk_size_tokens": 500,
  "chunk_overlap_tokens": 50,
  "distance": "cosine similarity via normalized IndexFlatIP",
  "embedding_dimension": 768,
  "number_of_vectors": 18142
}



# Stage A — Retrieve evidence for all 200 questions

Retrieval is performed **once** and saved. This has two advantages:

1. Every generation/evaluation stage sees exactly the same top-8 evidence.
2. The MedCPT query model can be unloaded before vLLM loads Qwen3-8B, freeing GPU memory.


In [ ]:
!pip install -q --force-reinstall --no-deps \
    "torchaudio==2.11.0+cu130" \
    --index-url https://download.pytorch.org/whl/cu130

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.6 MB/s eta 0:00:00


## Load MedCPT Query Encoder

In [ ]:
from transformers import AutoTokenizer, AutoModel

QUERY_MODEL_NAME = "ncbi/MedCPT-Query-Encoder"
TOP_K = 8

query_tokenizer = AutoTokenizer.from_pretrained(
    QUERY_MODEL_NAME
)

query_model = AutoModel.from_pretrained(
    QUERY_MODEL_NAME
).to("cuda")

query_model.eval()

if query_model.config.hidden_size != faiss_index.d:
    raise ValueError(
        f"MedCPT query dimension "
        f"{query_model.config.hidden_size} != "
        f"FAISS dimension {faiss_index.d}"
    )

print("Loaded:", QUERY_MODEL_NAME)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded: ncbi/MedCPT-Query-Encoder


## Retrieval functions

In [ ]:
@torch.inference_mode()
def embed_query(question: str) -> np.ndarray:
    encoded = query_tokenizer(
        [str(question)],
        truncation=True,
        padding=True,
        max_length=64,
        return_tensors="pt",
    )
    encoded = {k: v.to("cuda") for k, v in encoded.items()}

    output = query_model(**encoded)
    emb = output.last_hidden_state[:, 0, :]
    emb = emb.float().cpu().numpy().astype("float32")
    faiss.normalize_L2(emb)
    return emb


def retrieve_top_k(question: str, top_k: int = TOP_K):
    t0 = time.perf_counter()

    qvec = embed_query(question)
    scores, indices = faiss_index.search(qvec, top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        if idx < 0:
            continue

        m = dict(vector_metadata[idx])

        results.append({
            "rank": rank,
            "score": float(score),
            "faiss_id": int(idx),
            "row_id": m.get("row_id"),
            "chunk_id": m.get("chunk_id"),
            "question": m.get("question"),
            "focus_area": m.get("focus_area"),
            "question_type": m.get("question_type"),
            "source": m.get("source"),
            "text": m.get("text", ""),
        })

    latency = time.perf_counter() - t0
    return results, latency


def evidence_to_text(results):
    blocks = []
    for r in results:
        block = [
            f"[Chunk {r['rank']}]",
            f"Similarity score: {r['score']:.4f}",
        ]
        if r.get("question"):
            block.append(f"Original MedQuAD question: {r['question']}")
        if r.get("focus_area"):
            block.append(f"Focus area: {r['focus_area']}")
        if r.get("source"):
            block.append(f"Source: {r['source']}")
        block.append("Evidence:")
        block.append(str(r["text"]))
        blocks.append("\n".join(block))
    return "\n\n".join(blocks)

## Retrieve all questions and checkpoint to Drive

In [ ]:
# Set True to reuse retrievals already saved in Drive.
RESUME_RETRIEVAL = True

retrieval_records = []
retrieved_ids = set()

if RESUME_RETRIEVAL and RETRIEVAL_JSONL.exists():
    with open(RETRIEVAL_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                retrieval_records.append(rec)
                retrieved_ids.add(rec["ID"])

    print(f"Loaded {len(retrieval_records)} existing retrieval records.")

with open(RETRIEVAL_JSONL, "a", encoding="utf-8") as out:
    for _, row in tqdm(
        benchmark_df.iterrows(),
        total=len(benchmark_df),
        desc="Retrieving top-5 evidence",
    ):
        if row["ID"] in retrieved_ids:
            continue

        chunks, retrieval_latency = retrieve_top_k(row["Question"], TOP_K)

        rec = {
            **row.to_dict(),
            "retrieval_latency_s": float(retrieval_latency),
            "retrieved_chunks": chunks,
            "retrieved_context": evidence_to_text(chunks),
        }

        out.write(json.dumps(rec, ensure_ascii=False) + "\n")
        out.flush()
        retrieval_records.append(rec)

retrieval_df = pd.DataFrame(retrieval_records)
retrieval_df = retrieval_df.drop_duplicates("ID", keep="last")
retrieval_df = benchmark_df.merge(
    retrieval_df[
        ["ID", "retrieval_latency_s", "retrieved_chunks", "retrieved_context"]
    ],
    on="ID",
    how="left",
)

if retrieval_df["retrieved_context"].isna().any():
    raise RuntimeError("Some questions are missing retrieval results.")

print("Retrieval complete:", len(retrieval_df))
print(
    "Mean retrieval latency:",
    round(retrieval_df["retrieval_latency_s"].mean(), 4),
    "s",
)
display(
    retrieval_df[
        ["ID", "Question", "retrieval_latency_s"]
    ].head()
)

Loaded 200 existing retrieval records.


Retrieving top-5 evidence:   0%|          | 0/200 [00:00<?, ?it/s]

Retrieval complete: 200
Mean retrieval latency: 0.0978 s


,ID,Question,retrieval_latency_s
0,GND-001,What are the common symptoms of type 2 diabetes?,1.745147
1,GND-002,What symptoms are commonly associated with ast...,0.173846
2,GND-003,What are the usual signs and symptoms of psori...,0.130941
3,GND-004,What symptoms can occur in Parkinson's disease?,0.107358
4,GND-005,What are common symptoms of heart failure?,0.117739


### Unload MedCPT before loading vLLM

In [ ]:
del query_model
del query_tokenizer
gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated after unloading MedCPT:",
    round(torch.cuda.memory_allocated() / (1024**3), 3),
    "GB",
)

NameError: name 'query_model' is not defined


# Stage B — Generate all 200 RAG answers with Qwen3-8B

The notebook uses **`Qwen/Qwen3-8B-AWQ`**, the 4-bit AWQ version of Qwen3-8B, so the same 8B architecture is practical on common Colab GPUs.

**Thinking is disabled** during generation to make this the instruct/non-thinking condition.

Generation is performed sequentially so each question gets an interpretable per-query inference latency. Results are checkpointed after every answer.


## Generator configuration

In [ ]:
from vllm import LLM, SamplingParams

GENERATOR_MODEL = "Qwen/Qwen3-8B-AWQ"
GENERATOR_MAX_TOKENS = 512

generator_sampling = SamplingParams(
    temperature=0.0,
    max_tokens=GENERATOR_MAX_TOKENS,
)

RAG_SYSTEM_PROMPT = """You are a medical question-answering assistant.

Answer the user's question using the retrieved medical information.
Provide a clear and concise response.
"""


def make_generation_messages(question, retrieved_context):
    return [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": f"""Question:
{question}

Retrieved medical information:
{retrieved_context}

Answer the question.
""",
        },
    ]

## Load Qwen3-8B-AWQ in vLLM

In [ ]:
generator_llm = LLM(
    model=GENERATOR_MODEL,
    quantization="awq",
    dtype="half",
    trust_remote_code=True,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.88,
    max_model_len=8192,
    generation_config="vllm",
    seed=SEED,
)

print("Generator loaded:", GENERATOR_MODEL)

## Generate all answers and save them immediately to Drive

In [ ]:
# Set True to resume after a Colab interruption.
RESUME_GENERATION = True

generation_records = []
generated_ids = set()

if RESUME_GENERATION and GENERATION_JSONL.exists():
    with open(GENERATION_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                generation_records.append(rec)
                generated_ids.add(rec["ID"])

    print(f"Loaded {len(generation_records)} existing generations.")

retrieval_lookup = retrieval_df.set_index("ID").to_dict("index")

with open(GENERATION_JSONL, "a", encoding="utf-8") as out:
    for _, row in tqdm(
        benchmark_df.iterrows(),
        total=len(benchmark_df),
        desc="Generating Qwen3-8B RAG answers",
    ):
        qid = row["ID"]

        if qid in generated_ids:
            continue

        retrieved = retrieval_lookup[qid]
        messages = make_generation_messages(
            row["Question"],
            retrieved["retrieved_context"],
        )

        t0 = time.perf_counter()

        outputs = generator_llm.chat(
            messages,
            sampling_params=generator_sampling,
            use_tqdm=False,
            chat_template_kwargs={"enable_thinking": False},
        )

        generation_latency = time.perf_counter() - t0

        request_output = outputs[0]
        candidate = request_output.outputs[0]

        answer = candidate.text.strip()
        input_tokens = len(request_output.prompt_token_ids or [])
        output_tokens = len(candidate.token_ids or [])

        rec = {
            **row.to_dict(),
            "retrieval_latency_s": float(retrieved["retrieval_latency_s"]),
            "generation_latency_s": float(generation_latency),
            "end_to_end_latency_s": float(
                retrieved["retrieval_latency_s"] + generation_latency
            ),
            "input_tokens": int(input_tokens),
            "output_tokens": int(output_tokens),
            "retrieved_chunks": retrieved["retrieved_chunks"],
            "retrieved_context": retrieved["retrieved_context"],
            "generated_answer": answer,
            "generator_model": GENERATOR_MODEL,
        }

        out.write(json.dumps(rec, ensure_ascii=False) + "\n")
        out.flush()

        generation_records.append(rec)

print("Saved generation checkpoint:", GENERATION_JSONL)

Generating Qwen3-8B RAG answers:   0%|          | 0/200 [00:00<?, ?it/s]

Saved generation checkpoint: /content/drive/MyDrive/medical_rag_evaluation_200q/qwen3_8b_generation_results.jsonl
